In [6]:
from dotenv import load_dotenv
import os 

dotenv_path = os.path.join(os.getcwd(), '..', '.env')
load_dotenv (dotenv_path=dotenv_path)

API_KEY = os.getenv("GEMINI_API_KEY")
print(f"API Key loaded: {'Yes' if API_KEY else 'NO'}")
print(f"Working directory: {os.getcwd()}")

API Key loaded: Yes
Working directory: /Users/munizasiddiqui/easyf1/notebooks


In [7]:
#Importing FastF1 + enbale caching 
import fastf1
import pandas as pd 
import matplotlib.pyplot as plt
import warnings 

#supressing minoer warning that would clutter the output 
warnings.filterwarnings('ignore')

#Enable caching that is very important 
#Fastf1 download data from f1 servers 
#without cache, every time we run a cell it re downloads the same data, making it slow and uncessary
#with cache enable, data is locally saved on first download
#every run after that loads from the machine instantly 
fastf1.Cache.enable_cache('../cache')

print("FastF1 imported")
print (f"FastF1 version: {fastf1.__version__}")



Matplotlib is building the font cache; this may take a moment.


FastF1 imported
FastF1 version: 3.8.3


In [9]:
#loading f1 race session
#loading 2023 British GP race session 
# 'R' is Race 

print("Loading 2023 British GP... (first load takes 30 seconds)")

session = fastf1.get_session(2023, 'British Grand Prix', 'R')
session.load()
print(f"Session loaded")
#session.event is a dictionary containing metadata about the race weekend
print(f"Event: {session.event['EventName']}")
print(f"Circuit: {session.event['Location']}")
print(f"Date: {session.event['EventDate']}")
print(f"Total Laps: {session.total_laps}")

core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Loading 2023 British GP... (first load takes 30 seconds)


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '44', '81', '63', '11', '14', '23', '16', '55', '2', '77', '27', '18', '24', '22', '21', '10', '20', '31']


Session loaded
Event: British Grand Prix
Circuit: Silverstone
Date: 2023-07-09 00:00:00
Total Laps: 52


In [10]:
#loading all laps data for every single driver
laps = session.laps

print(f"Total rows in laps data: {len(laps)}")
print(f"Total columns available:{len(laps.columns)}")
print(f"All available columns:")
print(list(laps.columns))

Total rows in laps data: 971
Total columns available:31
All available columns:
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']


In [ ]:
#Narrowing the data for Hamilton only

hamilton_laps = laps[laps['Driver'] == 'HAM'] #boolean filtering 

print(f"Total Hamilton laps: {len(hamilton_laps)}")
print("\n Hamilton's Race Data: ")

#selecting only required or necessary columns, not all 31 columns 
strategy_cols = ['LapNumber', 'Compound', 'TyreLife', 'LapTime', 'Position', 'PitInTime', 'PitOutTime',
                 'TrackStatus']

print(hamilton_laps[strategy_cols].to_string(index = False))




Total Hamilton laps: 52

 Hamilton's Race Data: 
 LapNumber Compound  TyreLife                LapTime  Position              PitInTime             PitOutTime TrackStatus
       1.0   MEDIUM       1.0 0 days 00:01:40.664000       8.0                    NaT                    NaT           1
       2.0   MEDIUM       2.0 0 days 00:01:34.361000       8.0                    NaT                    NaT           1
       3.0   MEDIUM       3.0 0 days 00:01:33.662000       8.0                    NaT                    NaT           1
       4.0   MEDIUM       4.0 0 days 00:01:33.086000       8.0                    NaT                    NaT           1
       5.0   MEDIUM       5.0 0 days 00:01:33.718000       8.0                    NaT                    NaT           1
       6.0   MEDIUM       6.0 0 days 00:01:34.556000       8.0                    NaT                    NaT           1
       7.0   MEDIUM       7.0 0 days 00:01:33.328000       7.0                    NaT                   

In [12]:
#Finding Hamiltion's pitstop
#PitInTime is the value when car entered the pit lane 
#Filtering for laps where PitInTime in not NAT
hamilton_all = session.laps.pick_drivers('HAM')
pit_laps = hamilton_all[hamilton_all['PitInTime'].notna()]

print('Hamilton pit stops: ')
print(pit_laps[['LapNumber', 'Compound', 'TyreLife', 'LapTime', 'Position', 'PitInTime', 'PitOutTime',
                 'TrackStatus']].to_string(index = False))

Hamilton pit stops: 
 LapNumber Compound  TyreLife                LapTime  Position              PitInTime PitOutTime TrackStatus
      33.0   MEDIUM      33.0 0 days 00:02:03.496000       3.0 0 days 01:53:57.539000        NaT          64


In [13]:
#Hamilton's whole race strategy
print("Hamilton's complete race strategy")

#Group by stint number
#stint is one continous run on a set of tyres 
#stint = 1, lap 1 to first pit stop 
#stint = 2, lap after 1st pit to second pit

for stint in sorted(hamilton_all['Stint'].unique()):
    #getting all laps in this stint
    stint_laps = hamilton_all[hamilton_all['Stint'] == stint]
    #getting key info about this stint
    compound = stint_laps['Compound'].iloc[0]
    first_lap = stint_laps['LapNumber'].min()
    last_lap = stint_laps['LapNumber'].max()
    tyre_life = stint_laps['TyreLife'].max()
    avg_lap = stint_laps["LapTime"].mean()

    print(f"Stint {int(stint)}: ")
    print(f"Compound: {compound}")
    print(f"Laps: {int(first_lap)} {int(last_lap)}")
    print(f"Tyre Life: {int(tyre_life)} laps")
    print(f" Avg lap; {avg_lap}")

Hamilton's complete race strategy
Stint 1: 
Compound: MEDIUM
Laps: 1 33
Tyre Life: 33 laps
 Avg lap; 0 days 00:01:34.378484848
Stint 2: 
Compound: SOFT
Laps: 34 52
Tyre Life: 22 laps
 Avg lap; 0 days 00:01:43.193555555


In [14]:
#checking the tyre life lap by lap for stint 2
stint2 = hamilton_all[hamilton_all['Stint'] == 2]

print("Stint 2 lap by lap tyre life ")
print(stint2[['LapNumber', 'Compound', 'TyreLife', 'FreshTyre']].to_string(index=False))

Stint 2 lap by lap tyre life 
 LapNumber Compound  TyreLife  FreshTyre
      34.0     SOFT       4.0      False
      35.0     SOFT       5.0      False
      36.0     SOFT       6.0      False
      37.0     SOFT       7.0      False
      38.0     SOFT       8.0      False
      39.0     SOFT       9.0      False
      40.0     SOFT      10.0      False
      41.0     SOFT      11.0      False
      42.0     SOFT      12.0      False
      43.0     SOFT      13.0      False
      44.0     SOFT      14.0      False
      45.0     SOFT      15.0      False
      46.0     SOFT      16.0      False
      47.0     SOFT      17.0      False
      48.0     SOFT      18.0      False
      49.0     SOFT      19.0      False
      50.0     SOFT      20.0      False
      51.0     SOFT      21.0      False
      52.0     SOFT      22.0      False


In [15]:
#Extracting safety car and and VSC periods, for strategy analysis
print("Track status codes in the race:")
print(laps['TrackStatus'].unique())
print()

#defining what each status codes mean for us
sc_codes = ['4', '41', '42', '44', '45', '46', '47', '48']
vsc_codes = ['6', '61', '62', '64', '65', '66', '67', '68']

def get_track_events (laps_data):
    """
    Identifies sc and vsc periods in a race
    Returns memory of when each periods end and start

    Parameters:
    laps_data = DataFrame

    full laps DataFrame from FastF1

    Returns: 
    events: list of dicts 
    Each dicts denotes one sc or vsc period

    """

    events = []
    #using drop_duplicates to get one row per lap 
    laps_status = laps_data [['TrackStatus', 'LapNumber']].drop_duplicates(subset = 'LapNumber').sort_values('LapNumber')

    current_event = None

    for _, row in laps_status.iterrows():
        lap = row['LapNumber']
        status = str(row['TrackStatus'])

        #checking if any sc code is in the status string
        is_sc = any(code in status for code in sc_codes)

        #checking if any vsc code is in the status string
        is_vsc = any(code in status for code in vsc_codes )

        if is_sc and current_event is None:
            #sc just started
            current_event = {'type': 'Safety car', 'start_lap':  lap }

        elif is_vsc and current_event is None:
            #VSC just started
            current_event = {'type': 'VSC', 'start_lap': lap}

        elif not is_sc and not is_vsc and current_event is not None:
            #sc/vsc just ended
            current_event ['end_lap'] = lap - 1
            current_event ['duration'] = int(current_event['end_lap'] - current_event['start_lap'] + 1)

            events.append(current_event)
            current_event = None

    #case where sc was still active at the race end
    if current_event is not None:
        current_event ['end_lap'] = laps_status['LapNumber'].max()
        events.append(current_event)

    return events

events = get_track_events(laps)
print("Safety car and VSC periods in 2023 British GP:")
print("="*50)
for e in events:
    print(f"  {e['type']}")
    print(f"  Start lap : {int(e['start_lap'])}")
    print(f"  End lap   : {int(e['end_lap'])}")
    print(f"  Duration  : {e['duration']} laps")
    print()





Track status codes in the race:
['1' '1264' '4' '41' '126' '64']

Safety car and VSC periods in 2023 British GP:
  Safety car
  Start lap : 33
  End lap   : 38
  Duration  : 6 laps



In [16]:
#Improved track event detection
def get_track_events_v2(laps_data):
    """"Improved version that identifies both SC and VSC period including overlapping statuses"""

    #Getting one row per lap with it track position
    lap_status = laps_data[['LapNumber', 'TrackStatus']]\
        .drop_duplicates(subset = 'LapNumber')\
        .sort_values('LapNumber')
    
    
    print("Lap by lap track status around the incident:")
    print("showing laps 28-42 only")
    print()

    #printing lap by lap  status are the window 
    incident_window = lap_status [(lap_status["LapNumber"] >= 28) & (lap_status["LapNumber"] <= 42)]

    for _, row in incident_window.iterrows():
        lap = int(row['LapNumber'])
        status = str(row['TrackStatus'])    

        #Decode what is happening 
        if '4' in status and '6' in status:
            meaning = 'SC + VSC transition'
        elif '4' in status:
            meaning = 'Safety Car'
        elif '6' in status:
            meaning = 'Virtual Safety Car'
        elif status == '1':
            meaning = 'Green Flag'
        else: 
            meaning = f"Mixed flags"

        print(f" Lap {lap:>2}  | Status: {status:<6} | {meaning}")

get_track_events_v2(laps)

Lap by lap track status around the incident:
showing laps 28-42 only

 Lap 28  | Status: 1      | Green Flag
 Lap 29  | Status: 1      | Green Flag
 Lap 30  | Status: 1      | Green Flag
 Lap 31  | Status: 1      | Green Flag
 Lap 32  | Status: 1      | Green Flag
 Lap 33  | Status: 1264   | SC + VSC transition
 Lap 34  | Status: 4      | Safety Car
 Lap 35  | Status: 4      | Safety Car
 Lap 36  | Status: 4      | Safety Car
 Lap 37  | Status: 4      | Safety Car
 Lap 38  | Status: 41     | Safety Car
 Lap 39  | Status: 1      | Green Flag
 Lap 40  | Status: 1      | Green Flag
 Lap 41  | Status: 1      | Green Flag
 Lap 42  | Status: 1      | Green Flag


In [ ]:
#converting race data into RAG text chunks 
#this is the bridge between raw f1 data and the AI system
#RAG works by sotring text that LLM can read and reason over 
#Thats why we are comverting our dataframes into meaningful sentences
#that describes what happened at each lap 

DRIVER_NAMES ={
    'HAM' : 'Lewis Hamilton',
    'VER' : 'Max Verstappen',
    'LEC' : 'Charles Leclerc',
    'PER' : 'Sergio Perez',
    'RUS' : 'George Russell',
    'SAI' : 'Carlos Sainz',
    'GAS' : 'Pierre Gasly',
    'NOR' : 'Lando Norris',
    'OCO' : 'Esteban Ocon',
    'ALO' : 'Fernando Alonso',
    'STR' : 'Lance Stroll',
    'BOT' : 'Valtteri Bottas',
    'ZHO' : 'Zho Guanyu',
    'MAG' : 'Kevin Magnussen',
    'HUL' : 'Nico Hulkenberg',
    'DEV' : 'Nyck de Vries',
    'ALB' : 'Alex Albon',
    'SAR' : 'Logan Sargeant',
    'PIA' : 'Oscar Piastri'
}

def create_race_chunks(laps_data, session):
    """
    Converts a full race session into text chunks 
    suitable to store in a RAG vector database.

    each chunk describes the strategic situation at a specific lap
    from a specific drivers perspective 

    parameters: 
    session: FastF1 ession object, the loaded race session
    laps_data: DataFrame, full laps dataframe from FastF1 

    Returns: 
    chunks: list of dicts
    each dicts has 'text'(chunks) and metadata (structured data for filtering)

    """
    chunks = []

    #Get race info
    race_name = session.event['EventName']
    race_year = session.event['EventDate'].year
    circuit = session.event['Location']

    #Get track status per lap (one row per lap)
    lap_status = laps_data[['LapNumber', 'TrackStatus']]\
        .drop_duplicates(subset = 'LapNumber')\
        .sort_values('LapNumber')

    #creating a lookup dictionary: lap number -> track status 
    # this way we can check status for any lap 

    status_lookup = dict(zip(
        lap_status['LapNumber'],
        lap_status['TrackStatus']
    ))

    #Decoding track status into plain english
    def decode_status(status_code):
        s = str(status_code)

        if '4' in s:
            return " Safety car deployed"
        elif '6' in s:
            return "VSC deployed"
        elif s == '1':
            return " Green flag racing"
        else: 
            return "Mixed flags/ incident"
    
    #Getting list of all the drivers in the race 
    drivers = laps_data['Driver'].unique()

    print(f"Creating chunks for {race_year}, {race_name}...")
    print(f"Processing {len(drivers)} drivers across {int(laps_data['LapNumber'].max())} laps")
    print()


    #Loopin through every driver 
    for driver in drivers:

        #Getting ths drivers laps 
        driver_laps = laps_data[laps_data['Driver'] == driver].copy()

        #Get team name
        team = driver_laps['Team'].iloc[0] if len(driver_laps) > 0 else 'Unknown'

        #Looping through every lap from this driver 
        for _, lap in driver_laps.iterrows():

            lap_num = lap['LapNumber']
            compound = lap['Compound']
            tyre_age = lap['TyreLife']
            position = lap['Position']
            lap_time = lap['LapTime']
            stint = lap['Stint']

            #Getting track status for this lap 
            track_status = status_lookup.get(lap_num, '1')
            status_text = decode_status(track_status)

            #checking if this lap had a pit stop
            has_pit_in = not pd.isna(lap['PitInTime'])
            has_pit_out = not pd.isna(lap['PitOutTime'])

            pit_text = ""
            if has_pit_in:
                pit_text = f"{driver}  pitted at the end of this lap"
            
            if has_pit_out:
                pit_text = f"{driver} rejoined after pitting"

            #Formatting lap time cleanly
            if pd.isna (lap_time):
                lap_time_text = "No lap time recorded"
            else: 
                total_seconds = lap_time.total_seconds()
                minutes = int(total_seconds // 60)
                seconds = total_seconds % 60 
                lap_time_text = f"{minutes}: {seconds:05.2f}"
            
            #Building the chunk text
            #This is what gets store in the chromaDB and retrieved when agent need context

            full_name = DRIVER_NAMES.get(driver, driver)

            chunk_text = (
                f"Race {race_year} {race_name} at {circuit}. "
                f"Lap {int(lap_num)} of {int(laps_data['LapNumber'].max())}. "
                f"Driver: {full_name} ({driver}) racing for {team}. "
                f"Running in position {int(position) if not pd.isna(position) else 'Unkown'}. "
                f"Tyre Compound: {compound}, " 
                f"tyre_age: {int(tyre_age) if not pd.isna(tyre_age) else 'Unknown'} laps. "
                f"Stint number: {int(stint)}. "
                f"Track condition: {status_text}. "
                f"Lap time: {lap_time_text}. "
                f"{pit_text}"
            )

            #Metadata stored along side the chunk 
            #Will be used for filtering in ChromaDB later 
            metadata = {
                'race'       :  f"{race_year}, {race_name}",
                'circuit'    : circuit,
                'year'       : race_year,
                'lap'        : int(lap_num),
                'driver'     : driver,
                'team'       : team,
                'compound'   : str(compound),
                'tyre_age'   : int(tyre_age) if not pd.isna(tyre_age) else 0,
                'position'   : int(position) if not pd.isna(position) else 0,
                'sc_active'  : '4' in str(track_status),
                'vsc_active' : '6' in str(track_status),
                'had_pit'    : has_pit_in,
            }

            chunks.append({
                'text' : chunk_text,
                'metadata' : metadata
            })

    print(f"total chunks created: {len(chunks)}")
    return chunks 

chunks = create_race_chunks(laps,session)











Creating chunks for 2023, British Grand Prix...
Processing 20 drivers across 52 laps

total chunks created: 971


In [18]:
#verifying a chunk 
#chunk for hamilton in lap 33
ham_lap33 = [
    chunk for chunk in chunks
    if chunk ['metadata'] ['driver'] == 'HAM' 
    and chunk ['metadata'] ['lap'] == 33
]

print ('Ham lap 33 chunk')
print ('Text:')
print(ham_lap33[0]['text'])
print()
print('Metadata:')
print()
for key, value in ham_lap33 [0]['metadata'].items():
    print (f"{key}: {value}")

Ham lap 33 chunk
Text:
Race 2023 British Grand Prix at Silverstone. Lap 33 of 52. Driver: Lewis Hamilton (HAM) racing for Mercedes. Running in position 3. Tyre Compound: MEDIUM, tyre_age: 33 laps. Stint number: 1. Track condition:  Safety car deployed. Lap time: 2: 03.50. HAM  pitted at the end of this lap

Metadata:

race: 2023, British Grand Prix
circuit: Silverstone
year: 2023
lap: 33
driver: HAM
team: Mercedes
compound: MEDIUM
tyre_age: 33
position: 3
sc_active: True
vsc_active: True
had_pit: True


In [21]:
#Save chunks to CSV
#Saving the chunks to disk so we dont have to regenrate them everytime we restart the notebook
#Creating it as RAG knowledge base file

import csv 
import os 

#creating a data folder inside easyf1 
data_folder = os.path.join(os.getcwd(), '..', 'data')
os.makedirs(data_folder, exist_ok = True)

#definig the file path 
output_path = os.path.join(data_folder, '2023_british_gp_chunks.csv')

#saving chunks to csv 
with open(output_path, 'w', newline = '', encoding = 'utf-8') as f: 
    writer = csv.DictWriter(f, fieldnames = [
        'text', 'race', 'circuit', 'year', 'lap', 'driver', 'team',
        'compound', 'tyre_age', 'position', 'sc_active', 'vsc_active', 'had_pit'
    ])
    writer.writeheader()

    for chunk in chunks:
        row = {'text': chunk['text']}
        row.update(chunk['metadata'])
        writer.writerow(row)
print(f"chunks saved successfully")
print(f"Location: {output_path}")
print(f"Total chunks save: {len(chunks)}")


chunks saved successfully
Location: /Users/munizasiddiqui/easyf1/notebooks/../data/2023_british_gp_chunks.csv
Total chunks save: 971
